**Status: parallel research track.**

This is a parallel, **stint-level** exploration of the same problem `phase_3_ranking.ipynb` solves at
lap-level. Instead of podium/win probability, it tries to reconstruct a full pit strategy
(tyre compound + stint lengths) via three separate models: a compound classifier, a
next-pit-lap regressor, and a win-probability classifier, feeding `predict_strategy(...)`.

It reads from the same `workspace.default.f1_cleaned_lap_dataset` table as `phase_3_ranking.ipynb`,
and its artifacts (`m1_compound_classifier.json`, `m2_pitlap_regressor.json`,
`m3_win_classifier.json`, `preproc_pipeline/`) are consumed by `notebooks/phase_4.ipynb` and the
Databricks App — but independently of `phase_3_ranking.ipynb`'s `pre_model`/`live_model`. Treat this
notebook as a separate research track: changes here don't need to be mirrored in `phase_3_ranking.ipynb`,
and vice versa.

# Phase 3 — F1 Pit Stop Strategy Prediction

Train XGBoost models (via Spark) to predict **optimal pit stop strategy**:
- Which **tyre compound** to use each stint (classification)
- Which **lap to pit** on (regression)
- **Win probability** with that strategy

| Split | Years |
|-------|-------|
| Train | ≤ 2020 |
| Validation | 2021 |
| Test | ≥ 2022 |


## 1. Environment Setup

In [0]:
import warnings, os, sys
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Spark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, RegressionEvaluator

# XGBoost
import xgboost as xgb
SPARK_XGB = False
print("xgboost.spark not available – will use xgboost + pandas bridge")

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, confusion_matrix,
    classification_report, mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.preprocessing import LabelEncoder

plt.rcParams["figure.dpi"] = 110
sns.set_theme(style="darkgrid", palette="muted")
print("Environment OK")


In [0]:
spark = (
    SparkSession.builder
    .appName("F1_PitStop_Strategy")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
#spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)


## 2. Load Data

In [0]:
DATA_PATH = "workspace.default.f1_cleaned_lap_dataset"

sdf = spark.read.option("header", True).option("inferSchema", True).table(DATA_PATH)
print(f"Rows: {sdf.count():,}  |  Cols: {len(sdf.columns)}")
sdf.printSchema()


In [0]:
pdf = sdf.toPandas()
print("Years:", sorted(pdf["Year"].unique()))
print("Compounds:", sorted(pdf["Compound"].unique()))
print("Circuits:", sorted(pdf["Circuit"].unique()))
pdf.head(3)


## 3. Exploratory Data Analysis

In [0]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Compound distribution
comp_vc = pdf["Compound"].value_counts()
axes[0,0].bar(comp_vc.index, comp_vc.values, color="steelblue")
axes[0,0].set_title("Compound Usage Distribution"); axes[0,0].tick_params(axis="x", rotation=30)

# Final position histogram
race_res = pdf.drop_duplicates(["Driver","Year","Circuit"])
axes[0,1].hist(race_res["FinalPosition"].dropna(), bins=20, color="coral", edgecolor="k")
axes[0,1].set_title("Final Position Distribution")

# Pit count per race
pit_cnts = pdf[pdf["HasPit"]==1].groupby(["Driver","Year","Circuit"]).size()
axes[0,2].hist(pit_cnts, bins=10, color="mediumseagreen", edgecolor="k")
axes[0,2].set_title("Pit Stops per Race")

# LapTime by compound
pdf_c = pdf[pdf["LapTime"] < pdf["LapTime"].quantile(0.98)]
medians = pdf_c.groupby("Compound")["LapTime"].median().sort_values()
axes[1,0].barh(medians.index, medians.values, color="salmon")
axes[1,0].set_title("Median Lap Time by Compound (s)")

# Tyre degradation
for comp, col in [("SOFT","#e74c3c"),("MEDIUM","#f39c12"),("HARD","#2c3e50")]:
    sub = pdf_c[pdf_c["Compound"].isin(["SOFT","SUPERSOFT","ULTRASOFT"] if comp=="SOFT" else [comp])]
    deg = sub.groupby("TyreLife")["LapTime"].mean()
    axes[1,1].plot(deg.index[:40], deg.values[:40], label=comp, color=col)
axes[1,1].set_title("Tyre Degradation"); axes[1,1].set_xlabel("Tyre Age (laps)"); axes[1,1].legend()

# Quali vs Race pos
rr2 = race_res[["QualiPosition","FinalPosition"]].dropna()
axes[1,2].scatter(rr2["QualiPosition"], rr2["FinalPosition"], alpha=0.25, color="purple")
axes[1,2].set_title("Qualifying vs Race Position"); axes[1,2].set_xlabel("Quali"); axes[1,2].set_ylabel("Race")

plt.tight_layout(); plt.show()


## 4. Feature Engineering

### 4.1 Canonical Compound Mapping

Map all Pirelli compound names (ULTRASOFT, SUPERSOFT, SOFT → SOFT; MEDIUM; HARD; INTERMEDIATE; WET)
to 5 canonical classes.


In [0]:
sdf = sdf.withColumn(
    "CompoundCanon",
    F.when(F.col("Compound").isin(["ULTRASOFT","SUPERSOFT","SOFT"]), "SOFT")
     .when(F.col("Compound") == "MEDIUM", "MEDIUM")
     .when(F.col("Compound") == "HARD", "HARD")
     .when(F.col("Compound") == "INTERMEDIATE", "INTERMEDIATE")
     .when(F.col("Compound") == "WET", "WET")
     .otherwise("SOFT")
).withColumn(
    "CompoundRank",
    F.when(F.col("CompoundCanon")=="WET", 0)
     .when(F.col("CompoundCanon")=="INTERMEDIATE", 1)
     .when(F.col("CompoundCanon")=="HARD", 2)
     .when(F.col("CompoundCanon")=="MEDIUM", 3)
     .otherwise(4)  # SOFT fastest
)
print("Compound mapping OK")


### 4.2 Race-level windows (stint count, lap totals)

In [0]:
race_win  = Window.partitionBy("Driver","Year","Circuit")
stint_win = Window.partitionBy("Driver","Year","Circuit","Stint")

sdf = (sdf
    .withColumn("TotalLaps",    F.max("LapNumber").over(race_win))
    .withColumn("NumStints",    F.max("Stint").over(race_win))
    .withColumn("StintAvgLT",   F.mean("LapTime").over(stint_win))
    .withColumn("StintLapCount",F.count("LapNumber").over(stint_win))
    .withColumn("LapFrac",      F.col("LapNumber") / F.col("TotalLaps"))
)
print("Window features OK")


### 4.3 Driver Performance Profile (Skill)

Aggregated **per driver per season** so it captures: 
- `DriverAvgPos` – average finishing position (lower = better)
- `DriverAvgQuali` – average qualifying position (grid pace)  
- `DriverWinRate` – win percentage  
- `DriverPointsRate` – top-10 finish percentage  
- `DriverConsistency` – std-dev of finishing position (lower = more consistent)


In [0]:
race_level = sdf.select(
    "Driver","Year","Circuit","TeamName","FinalPosition","QualiPosition"
).dropDuplicates(["Driver","Year","Circuit"])

driver_profile = race_level.groupBy("Driver","Year").agg(
    F.mean("FinalPosition").alias("DriverAvgPos"),
    F.stddev("FinalPosition").alias("DriverConsistency"),
    F.mean("QualiPosition").alias("DriverAvgQuali"),
    F.count("Circuit").alias("DriverRaceCount"),
    (F.sum(F.when(F.col("FinalPosition")==1,1).otherwise(0)) / F.count("Circuit")).alias("DriverWinRate"),
    (F.sum(F.when(F.col("FinalPosition")<=10,1).otherwise(0)) / F.count("Circuit")).alias("DriverPointsRate"),
    (F.sum(F.when(F.col("FinalPosition")<=3,1).otherwise(0)) / F.count("Circuit")).alias("DriverPodiumRate"),
)

sdf = sdf.join(driver_profile, on=["Driver","Year"], how="left")
print("Driver profile joined")


### 4.4 Constructor (Car) Performance Profile

Aggregated **per team per season** – captures car competitiveness which varies year-to-year:
- `CarAvgPos` – average team finishing position  
- `CarWinRate` – team win rate  
- `CarPodiumRate` – team podium rate  
- `CarAvgQuali` – team average grid position (raw speed proxy)


In [0]:
constructor_profile = race_level.groupBy("TeamName","Year").agg(
    F.mean("FinalPosition").alias("CarAvgPos"),
    F.mean("QualiPosition").alias("CarAvgQuali"),
    F.count("Circuit").alias("CarRaceCount"),
    (F.sum(F.when(F.col("FinalPosition")==1,1).otherwise(0)) / F.count("Circuit")).alias("CarWinRate"),
    (F.sum(F.when(F.col("FinalPosition")<=3,1).otherwise(0)) / F.count("Circuit")).alias("CarPodiumRate"),
    (F.sum(F.when(F.col("FinalPosition")<=10,1).otherwise(0)) / F.count("Circuit")).alias("CarPointsRate"),
)

sdf = sdf.join(constructor_profile, on=["TeamName","Year"], how="left")
print("Constructor profile joined")


### 4.4b Starting Grid Context (Threat Score)

In [ ]:
# Compute Threat Score: sum of DriverWinRate for all drivers starting ahead 
# of the current driver on the grid.
# First, create a mapping of Race -> GridPosition -> DriverWinRate
grid_rates = sdf.select("Year", "Circuit", "QualiPosition", "DriverWinRate").dropDuplicates()
grid_rates = grid_rates.withColumnRenamed("QualiPosition", "AheadQualiPosition") \
                       .withColumnRenamed("DriverWinRate", "AheadWinRate")

# Self-join to find all drivers ahead (AheadQualiPosition < QualiPosition)
threat_df = sdf.select("Driver", "Year", "Circuit", "QualiPosition").dropDuplicates()
threat_joined = threat_df.join(
    grid_rates,
    on=["Year", "Circuit"],
    how="left"
).filter(F.col("AheadQualiPosition") < F.col("QualiPosition"))

# Aggregate the threat score (sum of win rates of drivers ahead)
threat_agg = threat_joined.groupBy("Driver", "Year", "Circuit").agg(
    F.sum("AheadWinRate").alias("ThreatScoreAhead"),
    F.max("AheadWinRate").alias("MaxThreatAhead")
).fillna(0.0)

sdf = sdf.join(threat_agg, on=["Driver", "Year", "Circuit"], how="left")
sdf = sdf.fillna({"ThreatScoreAhead": 0.0, "MaxThreatAhead": 0.0})
print("Threat Score Context added")


### 4.5 Circuit Tyre Tendencies

In [0]:
circuit_profile = sdf.groupBy("Circuit").agg(
    F.mean("CompoundRank").alias("CircuitAvgCompoundRank"),
    F.mean("NumStints").alias("CircuitAvgStints"),
    F.mean("TrackTemp_C").alias("CircuitAvgTrackTemp"),
)
sdf = sdf.join(circuit_profile, on="Circuit", how="left")
print("Circuit profile joined")


## 5. Build Stint-Level Dataset

One row per stint. Target variables:
1. `StintCompound` → **compound classification** target  
2. `NextPitLap` → **pit lap regression** target  
3. `IsWin` → **win probability** classification target


In [0]:
stint_sdf = sdf.groupBy("Driver","Year","Circuit","TeamName","Stint").agg(
    F.first("CompoundCanon").alias("StintCompound"),
    F.first("CompoundRank").alias("StintCompoundRank"),
    F.min("LapNumber").alias("StintStartLap"),
    F.max("LapNumber").alias("StintEndLap"),
    F.count("LapNumber").alias("StintLength"),
    F.mean("LapTime").alias("StintAvgLapTime"),
    F.min("LapTime").alias("StintBestLap"),
    F.stddev("LapTime").alias("StintLapTimeStd"),
    F.first("FreshTyre").alias("FreshTyre"),
    F.first("FinalPosition").alias("FinalPosition"),
    F.first("QualiPosition").alias("QualiPosition"),
    F.first("TotalLaps").alias("TotalLaps"),
    F.first("NumStints").alias("NumStints"),
    F.mean("GapToAhead").alias("AvgGapToAhead"),
    F.mean("DeltaToLeader").alias("AvgDeltaToLeader"),
    F.last("Position").alias("PositionAtStintEnd"),
    F.first("AirTemp_C").alias("AirTemp_C"),
    F.first("TrackTemp_C").alias("TrackTemp_C"),
    F.first("Humidity_pct").alias("Humidity_pct"),
    F.first("WindSpeed_kmh").alias("WindSpeed_kmh"),
    F.first("DriverAvgPos").alias("DriverAvgPos"),
    F.first("DriverAvgQuali").alias("DriverAvgQuali"),
    F.first("DriverWinRate").alias("DriverWinRate"),
    F.first("DriverPointsRate").alias("DriverPointsRate"),
    F.first("DriverPodiumRate").alias("DriverPodiumRate"),
    F.first("DriverConsistency").alias("DriverConsistency"),
    F.first("CarAvgPos").alias("CarAvgPos"),
    F.first("CarAvgQuali").alias("CarAvgQuali"),
    F.first("CarWinRate").alias("CarWinRate"),
    F.first("CarPodiumRate").alias("CarPodiumRate"),
    F.first("CarPointsRate").alias("CarPointsRate"),
    F.first("CircuitAvgCompoundRank").alias("CircuitAvgCompoundRank"),
    F.first("CircuitAvgStints").alias("CircuitAvgStints"),
    F.first("CircuitAvgTrackTemp").alias("CircuitAvgTrackTemp"),
    F.first("ThreatScoreAhead").alias("ThreatScoreAhead"),
    F.first("MaxThreatAhead").alias("MaxThreatAhead"),
)

stint_order = Window.partitionBy("Driver","Year","Circuit").orderBy("Stint")
stint_sdf = stint_sdf     .withColumn("StintStartFrac", F.col("StintStartLap") / F.col("TotalLaps"))     .withColumn("NextPitLap", F.lead("StintStartLap").over(stint_order))     .withColumn("IsWin", F.when(F.col("FinalPosition")==1, 1).otherwise(0))

print("Stint dataset rows:", stint_sdf.count())
stint_sdf.select("Driver","Year","Circuit","Stint","StintCompound","StintStartLap","StintEndLap","NextPitLap").show(6)


## 6. Train / Validation / Test Split

| Split | Year(s) | Purpose |
|-------|---------|---------|
| Train | ≤ 2020 | Fit model weights |
| Validation | 2021 | Hyper-parameter tuning |
| Test | ≥ 2022 | Final held-out evaluation |


In [0]:
train_sdf = stint_sdf.filter(F.col("Year") < 2021)
val_sdf   = stint_sdf.filter(F.col("Year") == 2021)
test_sdf  = stint_sdf.filter(F.col("Year") > 2021)

print(f"Train  : {train_sdf.count():>6,} stints")
print(f"Val    : {val_sdf.count():>6,} stints")
print(f"Test   : {test_sdf.count():>6,} stints")


## 7. Feature Definitions

In [0]:
# ── Numeric features ────────────────────────────────────────────────────
NUMERIC_FEATURES = [
    # Stint context
    "Stint", "StintStartLap", "StintLength", "StintStartFrac",
    "StintAvgLapTime", "StintBestLap",
    # Race context
    "QualiPosition", "TotalLaps", "NumStints",
    "AvgGapToAhead", "AvgDeltaToLeader", "PositionAtStintEnd",
    # Weather
    "AirTemp_C", "TrackTemp_C", "Humidity_pct", "WindSpeed_kmh",
    # Driver profile (skill)
    "DriverAvgPos", "DriverAvgQuali", "DriverWinRate",
    "DriverPointsRate", "DriverPodiumRate", "DriverConsistency",
    # Constructor profile (car)
    "CarAvgPos", "CarAvgQuali", "CarWinRate",
    "CarPodiumRate", "CarPointsRate",
    # Circuit tendencies
    "CircuitAvgCompoundRank", "CircuitAvgStints", "CircuitAvgTrackTemp",
    # Grid context
    "ThreatScoreAhead", "MaxThreatAhead",
]

# ── Categorical features (to be indexed + OHE) ──────────────────────────
CAT_FEATURES = ["Circuit", "TeamName", "Driver"]

# ── Targets ──────────────────────────────────────────────────────────────
TARGET_COMPOUND  = "StintCompound"     # multi-class classification
TARGET_PIT_LAP   = "NextPitLap"        # regression
TARGET_WIN       = "IsWin"             # binary classification

print("Numeric features:", len(NUMERIC_FEATURES))
print("Categorical features:", CAT_FEATURES)
print("Targets:", TARGET_COMPOUND, "|", TARGET_PIT_LAP, "|", TARGET_WIN)


## 8. Spark ML Preprocessing Pipeline

In [0]:
# String indexers for categoricals
indexers = [
    StringIndexer(inputCol=c, outputCol=c+"_idx", handleInvalid="keep")
    for c in CAT_FEATURES
]
# One-hot encoders
encoders = [
    OneHotEncoder(inputCol=c+"_idx", outputCol=c+"_ohe")
    for c in CAT_FEATURES
]
# Compound label indexer (for classification target)
compound_indexer = StringIndexer(
    inputCol=TARGET_COMPOUND, outputCol="CompoundLabel", handleInvalid="keep"
)

# Assemble all numeric + OHE features
feature_cols = NUMERIC_FEATURES + [c+"_ohe" for c in CAT_FEATURES]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw", handleInvalid="keep")
scaler    = StandardScaler(inputCol="features_raw", outputCol="features", withMean=True, withStd=True)

preproc_pipeline = Pipeline(stages=indexers + encoders + [compound_indexer, assembler, scaler])

# Fit on train only!
preproc_model = preproc_pipeline.fit(train_sdf)
print("Preprocessing pipeline fitted ✓")

train_proc = preproc_model.transform(train_sdf)
val_proc   = preproc_model.transform(val_sdf)
test_proc  = preproc_model.transform(test_sdf)
print(f"Train: {train_proc.count():,}  Val: {val_proc.count():,}  Test: {test_proc.count():,}")


## 9. XGBoost Model Training

We train **three** XGBoost models:

| # | Model | Task | Target |
|---|-------|------|--------|
| M1 | Compound Classifier | Multi-class | Which tyre to use |
| M2 | Pit Lap Regressor | Regression | When to pit (lap number) |
| M3 | Win Probability Classifier | Binary | Will this driver win? |


In [0]:
# ─── Convert to pandas for xgboost if SparkXGBClassifier unavailable ────
def to_pandas_xy(spark_df, label_col, feature_col="features"):
    rows = spark_df.select(label_col, feature_col).dropna(subset=[label_col]).collect()
    X = np.array([r[feature_col].toArray() for r in rows])
    y = np.array([r[label_col] for r in rows])
    return X, y

if not SPARK_XGB:
    print("Extracting features to numpy arrays...")
    X_tr_raw, y_comp_tr = to_pandas_xy(train_proc, "CompoundLabel")
    X_va_raw, y_comp_va = to_pandas_xy(val_proc,   "CompoundLabel")
    X_te_raw, y_comp_te = to_pandas_xy(test_proc,  "CompoundLabel")

    # Pit lap - drop rows without NextPitLap (last stint has None)
    train_pit = train_proc.filter(F.col("NextPitLap").isNotNull())
    val_pit   = val_proc.filter(F.col("NextPitLap").isNotNull())
    test_pit  = test_proc.filter(F.col("NextPitLap").isNotNull())

    X_tr_pit, y_pit_tr = to_pandas_xy(train_pit, "NextPitLap")
    X_va_pit, y_pit_va = to_pandas_xy(val_pit,   "NextPitLap")
    X_te_pit, y_pit_te = to_pandas_xy(test_pit,  "NextPitLap")

    X_tr_win, y_win_tr = to_pandas_xy(train_proc, "IsWin")
    X_va_win, y_win_va = to_pandas_xy(val_proc,   "IsWin")
    X_te_win, y_win_te = to_pandas_xy(test_proc,  "IsWin")

    print(f"Compound arrays — Train:{X_tr_raw.shape}, Val:{X_va_raw.shape}, Test:{X_te_raw.shape}")
    print(f"Pit lap arrays  — Train:{X_tr_pit.shape}, Val:{X_va_pit.shape}, Test:{X_te_pit.shape}")


In [0]:
# ─── M1: Compound Classifier ─────────────────────────────────────────────
XGB_COMPOUND_PARAMS = dict(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.05,
    reg_lambda=1.0,
    use_label_encoder=False,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

if SPARK_XGB:
    m1 = SparkXGBClassifier(
        label_col="CompoundLabel", features_col="features",
        num_round=400, max_depth=6, eta=0.05,
        subsample=0.8, colsample_bytree=0.8,
        objective="multi:softprob",
        eval_metric="mlogloss",
        num_workers=1
    )
    m1_model = m1.fit(train_proc)
    val_preds_m1  = m1_model.transform(val_proc)
    test_preds_m1 = m1_model.transform(test_proc)
else:
    n_classes = int(y_comp_tr.max()) + 1
    m1 = xgb.XGBClassifier(num_class=n_classes, objective="multi:softprob", **XGB_COMPOUND_PARAMS)
    m1.fit(X_tr_raw, y_comp_tr,
           eval_set=[(X_va_raw, y_comp_va)],
           verbose=50)

print("M1 Compound Classifier trained ✓")


In [0]:
# ─── M2: Pit Lap Regressor ───────────────────────────────────────────────
XGB_PIT_PARAMS = dict(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.5,
    eval_metric="rmse",
    random_state=42,
    n_jobs=-1
)

if SPARK_XGB:
    m2 = SparkXGBRegressor(
        label_col="NextPitLap", features_col="features",
        num_round=400, max_depth=6, eta=0.05,
        subsample=0.8, colsample_bytree=0.8,
        objective="reg:squarederror",
        eval_metric="rmse",
        num_workers=1
    )
    train_pit_sp = train_proc.filter(F.col("NextPitLap").isNotNull())
    val_pit_sp   = val_proc.filter(F.col("NextPitLap").isNotNull())
    test_pit_sp  = test_proc.filter(F.col("NextPitLap").isNotNull())
    m2_model = m2.fit(train_pit_sp)
    val_preds_m2  = m2_model.transform(val_pit_sp)
    test_preds_m2 = m2_model.transform(test_pit_sp)
else:
    m2 = xgb.XGBRegressor(objective="reg:squarederror", **XGB_PIT_PARAMS)
    m2.fit(X_tr_pit, y_pit_tr,
           eval_set=[(X_va_pit, y_pit_va)],
           verbose=50)

print("M2 Pit Lap Regressor trained ✓")


In [0]:
# ─── M3: Win Probability Classifier ─────────────────────────────────────
XGB_WIN_PARAMS = dict(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.7,
    min_child_weight=10,
    gamma=0.2,
    scale_pos_weight=19,   # imbalanced: ~1/20 races have a winner
    reg_alpha=0.1,
    reg_lambda=1.0,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

if SPARK_XGB:
    m3 = SparkXGBClassifier(
        label_col="IsWin", features_col="features",
        num_round=300, max_depth=5, eta=0.05,
        subsample=0.8, colsample_bytree=0.7,
        objective="binary:logistic",
        scale_pos_weight=19,
        eval_metric="logloss",
        num_workers=1
    )
    m3_model = m3.fit(train_proc)
    val_preds_m3  = m3_model.transform(val_proc)
    test_preds_m3 = m3_model.transform(test_proc)
else:
    m3 = xgb.XGBClassifier(objective="binary:logistic", **XGB_WIN_PARAMS)
    m3.fit(X_tr_win, y_win_tr,
           eval_set=[(X_va_win, y_win_va)],
           verbose=50)

print("M3 Win Probability Classifier trained ✓")


## 10. Model Evaluation

### Evaluation Metrics

| Model | Primary Metrics | Secondary |
|-------|----------------|-----------|
| M1 Compound | Accuracy, F1-macro, Precision, Recall | Confusion matrix |
| M2 Pit Lap | MAE (laps), RMSE, R² | Error distribution |
| M3 Win Prob | AUC-ROC, F1, Precision@K | Calibration |

> **Why these metrics?**  
> - Compound: F1-macro handles class imbalance (WET/INTERMEDIATE rare).  
> - Pit Lap: MAE is interpretable (e.g. "off by 2.3 laps on average").  
> - Win: AUC + precision is critical because wins are rare (class=1 is ~5%).


In [0]:
def evaluate_classifier(y_true, y_pred, model_name, class_names=None):
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average="macro", zero_division=0)
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="macro", zero_division=0)
    print(f"\n{'='*55}")
    print(f"  {model_name}")
    print(f"{'='*55}")
    print(f"  Accuracy        : {acc:.4f} ({acc*100:.1f}%)")
    print(f"  F1 (macro)      : {f1:.4f}")
    print(f"  Precision (mac) : {prec:.4f}")
    print(f"  Recall (mac)    : {rec:.4f}")
    if class_names:
        print("\n", classification_report(y_true, y_pred, target_names=class_names, zero_division=0))
    return {"Accuracy": acc, "F1_macro": f1, "Precision_macro": prec, "Recall_macro": rec}

def evaluate_regressor(y_true, y_pred, model_name):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    r2   = r2_score(y_true, y_pred)
    within2 = (np.abs(y_true - y_pred) <= 2).mean()
    within5 = (np.abs(y_true - y_pred) <= 5).mean()
    print(f"\n{'='*55}")
    print(f"  {model_name}")
    print(f"{'='*55}")
    print(f"  MAE             : {mae:.3f} laps")
    print(f"  RMSE            : {rmse:.3f} laps")
    print(f"  R²              : {r2:.4f}")
    print(f"  Within ±2 laps  : {within2*100:.1f}%")
    print(f"  Within ±5 laps  : {within5*100:.1f}%")
    return {"MAE": mae, "RMSE": rmse, "R2": r2, "Within2Laps": within2, "Within5Laps": within5}


In [0]:
# ── Get label mapping ────────────────────────────────────────────────────
compound_labels = preproc_model.stages[len(CAT_FEATURES)*2].labels  # StringIndexer for compound
print("Compound label mapping:", {i: c for i, c in enumerate(compound_labels)})


In [0]:
# ── M1 Evaluation ─────────────────────────────────────────────────────
if SPARK_XGB:
    def spark_preds(pred_df, label_col, pred_col="prediction"):
        rows = pred_df.select(label_col, pred_col).dropna().collect()
        return np.array([r[label_col] for r in rows]), np.array([r[pred_col] for r in rows])

    y_true_m1_v, y_pred_m1_v = spark_preds(val_preds_m1,  "CompoundLabel")
    y_true_m1_t, y_pred_m1_t = spark_preds(test_preds_m1, "CompoundLabel")
else:
    y_pred_m1_v = m1.predict(X_va_raw)
    y_pred_m1_t = m1.predict(X_te_raw)
    y_true_m1_v, y_true_m1_t = y_comp_va, y_comp_te

m1_class_names = [compound_labels[i] for i in range(len(compound_labels))]
metrics_m1_val  = evaluate_classifier(y_true_m1_v.astype(int), y_pred_m1_v.astype(int), "M1 Compound Classifier — VALIDATION", m1_class_names)
metrics_m1_test = evaluate_classifier(y_true_m1_t.astype(int), y_pred_m1_t.astype(int), "M1 Compound Classifier — TEST",       m1_class_names)


In [0]:
# ── M1 Confusion Matrix ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, y_t, y_p, title in [
    (axes[0], y_true_m1_v.astype(int), y_pred_m1_v.astype(int), "Validation"),
    (axes[1], y_true_m1_t.astype(int), y_pred_m1_t.astype(int), "Test")
]:
    cm = confusion_matrix(y_t, y_p)
    sns.heatmap(cm, annot=True, fmt="d", ax=ax,
                xticklabels=m1_class_names, yticklabels=m1_class_names, cmap="Blues")
    ax.set_title(f"M1 Compound — {title}")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()


In [0]:
# ── M2 Evaluation ─────────────────────────────────────────────────────
if SPARK_XGB:
    y_true_m2_v, y_pred_m2_v = spark_preds(val_preds_m2,  "NextPitLap")
    y_true_m2_t, y_pred_m2_t = spark_preds(test_preds_m2, "NextPitLap")
else:
    y_pred_m2_v = m2.predict(X_va_pit)
    y_pred_m2_t = m2.predict(X_te_pit)
    y_true_m2_v, y_true_m2_t = y_pit_va, y_pit_te

metrics_m2_val  = evaluate_regressor(y_true_m2_v, y_pred_m2_v, "M2 Pit Lap Regressor — VALIDATION")
metrics_m2_test = evaluate_regressor(y_true_m2_t, y_pred_m2_t, "M2 Pit Lap Regressor — TEST")


In [0]:
# ── M2 Prediction Error Distribution ──────────────────────────────────
errors_v = y_pred_m2_v - y_true_m2_v
errors_t = y_pred_m2_t - y_true_m2_t

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(errors_v, bins=30, color="steelblue", edgecolor="k", alpha=0.8, label="Val")
axes[0].hist(errors_t, bins=30, color="coral", edgecolor="k", alpha=0.6, label="Test")
axes[0].axvline(0, color="k", linestyle="--"); axes[0].legend()
axes[0].set_title("Pit Lap Prediction Error (laps)"); axes[0].set_xlabel("Error (laps)")

axes[1].scatter(y_true_m2_v, y_pred_m2_v, alpha=0.3, color="steelblue", s=15)
mn, mx = min(y_true_m2_v.min(), y_pred_m2_v.min()), max(y_true_m2_v.max(), y_pred_m2_v.max())
axes[1].plot([mn,mx],[mn,mx],"k--", label="Perfect"); axes[1].legend()
axes[1].set_title("Predicted vs Actual Pit Lap — Val"); axes[1].set_xlabel("Actual"); axes[1].set_ylabel("Predicted")

axes[2].scatter(y_true_m2_t, y_pred_m2_t, alpha=0.3, color="coral", s=15)
axes[2].plot([mn,mx],[mn,mx],"k--"); axes[2].set_title("Predicted vs Actual Pit Lap — Test")
axes[2].set_xlabel("Actual"); axes[2].set_ylabel("Predicted")

plt.tight_layout(); plt.show()


In [0]:
# ── M3 Evaluation ─────────────────────────────────────────────────────
from sklearn.metrics import roc_auc_score, average_precision_score

if SPARK_XGB:
    def spark_proba(pred_df, label_col, prob_col="probability"):
        rows = pred_df.select(label_col, prob_col).dropna().collect()
        y_t  = np.array([r[label_col] for r in rows])
        y_p  = np.array([r[prob_col][1] for r in rows])   # prob of class=1
        return y_t, y_p
    y_true_m3_v, y_prob_m3_v = spark_proba(val_preds_m3,  "IsWin")
    y_true_m3_t, y_prob_m3_t = spark_proba(test_preds_m3, "IsWin")
else:
    y_prob_m3_v = m3.predict_proba(X_va_win)[:,1]
    y_prob_m3_t = m3.predict_proba(X_te_win)[:,1]
    y_true_m3_v, y_true_m3_t = y_win_va, y_win_te

for label, y_t, y_p in [("VALIDATION", y_true_m3_v, y_prob_m3_v), ("TEST", y_true_m3_t, y_prob_m3_t)]:
    auc  = roc_auc_score(y_t, y_p)
    ap   = average_precision_score(y_t, y_p)
    y_cls = (y_p >= 0.5).astype(int)
    f1   = f1_score(y_t, y_cls, zero_division=0)
    print(f"M3 Win Classifier — {label}")
    print(f"  AUC-ROC              : {auc:.4f}")
    print(f"  Avg Precision (AP)   : {ap:.4f}")
    print(f"  F1 (threshold=0.5)   : {f1:.4f}")
    print()


In [0]:
# ── M3 ROC Curve ──────────────────────────────────────────────────────
from sklearn.metrics import roc_curve, precision_recall_curve

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, y_t, y_p, label, color in [
    (axes[0], y_true_m3_v, y_prob_m3_v, "Validation", "steelblue"),
    (axes[0], y_true_m3_t, y_prob_m3_t, "Test", "coral"),
]:
    fpr, tpr, _ = roc_curve(y_t, y_p)
    auc = roc_auc_score(y_t, y_p)
    ax.plot(fpr, tpr, label=f"{label} AUC={auc:.3f}", color=color)
axes[0].plot([0,1],[0,1],"k--"); axes[0].set_title("Win Probability — ROC Curve")
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR"); axes[0].legend()

for y_t, y_p, label, color in [
    (y_true_m3_v, y_prob_m3_v, "Validation", "steelblue"),
    (y_true_m3_t, y_prob_m3_t, "Test", "coral"),
]:
    prec, rec, _ = precision_recall_curve(y_t, y_p)
    axes[1].plot(rec, prec, label=label, color=color)
axes[1].set_title("Win Probability — Precision-Recall Curve")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision"); axes[1].legend()
plt.tight_layout(); plt.show()


## 11. Feature Importance Analysis

In [0]:
if not SPARK_XGB:
    fig, axes = plt.subplots(1, 3, figsize=(22, 7))

    feat_names = NUMERIC_FEATURES + [f"{c}_ohe" for c in CAT_FEATURES]
    n = min(len(feat_names), m1.n_features_in_)
    names = feat_names[:n]

    for ax, model, title in [
        (axes[0], m1, "M1 — Compound Classifier"),
        (axes[1], m2, "M2 — Pit Lap Regressor"),
        (axes[2], m3, "M3 — Win Probability"),
    ]:
        imp = model.feature_importances_
        top15_idx = np.argsort(imp)[-15:]
        ax.barh([names[i] if i < len(names) else f"feat_{i}" for i in top15_idx],
                imp[top15_idx], color="steelblue")
        ax.set_title(title); ax.set_xlabel("Importance (gain)")

    plt.tight_layout()
    plt.savefig("feature_importance.png", bbox_inches="tight")
    plt.show()
else:
    print("Feature importance plot available when running with pandas XGBoost mode.")
    print("For SparkXGBClassifier, extract importances via model.get_feature_importances().")


## 12. Pit Stop Strategy Decoder

Given **driver, team, year, circuit, starting grid**, reconstruct the full strategy as:  
`[(lap_start, lap_end, compound), ...]`  
and report the **predicted win probability**.


In [0]:
COMPOUND_IDX_TO_NAME = {i: c for i, c in enumerate(compound_labels)}

# Hard limits on how many laps a tyre compound can physically survive
TYRE_LIFE_LIMITS = {
    "SOFT": 25,
    "MEDIUM": 40,
    "HARD": 55,
    "INTERMEDIATE": 30,
    "WET": 40
}

def predict_strategy(
    driver: str,
    team: str,
    year: int,
    circuit: str,
    quali_position: int,
    total_laps: int = 57,
    num_pit_stops: int = 1,  # User controls exactly how many stops
    verbose: bool = True
) -> dict:
    """
    Predict the optimal pit stop strategy for one driver in one race,
    enforcing realistic tyre degradation constraints and user-defined pit stops.

    Parameters
    ----------
    driver          : driver abbreviation (e.g. 'HAM')
    team            : team name (e.g. 'Mercedes')
    year            : race year
    circuit         : circuit name (e.g. 'Monza')
    quali_position  : grid start position
    total_laps      : total race laps (default 57)
    num_pit_stops   : exactly how many pit stops the driver will make
    verbose         : print strategy

    Returns
    -------
    dict with keys: strategy, win_probability
    """

    # ── Pull driver & constructor & circuit profile ─────────
    def get_profile(sdf_local, col, val, year_local):
        row = sdf_local.filter((F.col(col) == val) & (F.col("Year") == year_local)).first()
        if row is None: row = sdf_local.filter(F.col(col) == val).orderBy(F.desc("Year")).first()
        return row

    drv_row = get_profile(driver_profile, "Driver", driver, year)
    car_row = get_profile(constructor_profile, "TeamName", team, year)
    cir_row = circuit_profile.filter(F.col("Circuit") == circuit).first()

    def safe(row, key, default=10.0):
        try: return float(row[key]) if row and row[key] is not None else default
        except: return default

    drv_avg_pos   = safe(drv_row, "DriverAvgPos", 10.0)
    drv_avg_quali = safe(drv_row, "DriverAvgQuali", quali_position)
    drv_win_rate  = safe(drv_row, "DriverWinRate", 0.05)
    drv_pts_rate  = safe(drv_row, "DriverPointsRate", 0.5)
    drv_pod_rate  = safe(drv_row, "DriverPodiumRate", 0.15)
    drv_consist   = safe(drv_row, "DriverConsistency", 4.0)
    car_avg_pos   = safe(car_row, "CarAvgPos", 10.0)
    car_avg_quali = safe(car_row, "CarAvgQuali", quali_position)
    car_win_rate  = safe(car_row, "CarWinRate", 0.05)
    car_pod_rate  = safe(car_row, "CarPodiumRate", 0.15)
    car_pts_rate  = safe(car_row, "CarPointsRate", 0.5)
    cir_avg_comp  = safe(cir_row, "CircuitAvgCompoundRank", 3.0)
    cir_avg_stints= safe(cir_row, "CircuitAvgStints", 2.5)
    cir_avg_temp  = safe(cir_row, "CircuitAvgTrackTemp", 30.0)

    # ── Calculate Context/Threat Score ─────────
    # If the driver is P3, gather win rates for P1 and P2
    year_drivers = sdf.filter(F.col("Year") == year).select("Driver", "QualiPosition").dropDuplicates()
    threat_df = year_drivers.filter(F.col("QualiPosition") < quali_position)
    
    # Get win rates for threats
    threat_score = 0.0
    max_threat = 0.0
    for t_row in threat_df.collect():
        t_drv = get_profile(driver_profile, "Driver", t_row["Driver"], year)
        t_wr = safe(t_drv, "DriverWinRate", 0.0)
        threat_score += t_wr
        max_threat = max(max_threat, t_wr)

    strategy = []
    current_lap = 1
    current_pos = quali_position
    
    # User asks for N pit stops -> N + 1 stints
    total_stints = num_pit_stops + 1
    prev_compound = None

    for stint_num in range(1, total_stints + 1):
        lap_frac = current_lap / total_laps

        row_dict = {
            "Stint": stint_num,
            "StintStartLap": current_lap,
            "StintLength": max(1, total_laps - current_lap),
            "StintStartFrac": lap_frac,
            "StintAvgLapTime": 95.0,
            "StintBestLap": 92.0,
            "QualiPosition": quali_position,
            "TotalLaps": total_laps,
            "NumStints": total_stints,
            "AvgGapToAhead": 1.5,
            "AvgDeltaToLeader": max(0, (current_pos - 1) * 1.2),
            "PositionAtStintEnd": current_pos,
            "AirTemp_C": 22.0,
            "TrackTemp_C": cir_avg_temp,
            "Humidity_pct": 40.0,
            "WindSpeed_kmh": 2.0,
            "DriverAvgPos": drv_avg_pos,
            "DriverAvgQuali": drv_avg_quali,
            "DriverWinRate": drv_win_rate,
            "DriverPointsRate": drv_pts_rate,
            "DriverPodiumRate": drv_pod_rate,
            "DriverConsistency": drv_consist,
            "CarAvgPos": car_avg_pos,
            "CarAvgQuali": car_avg_quali,
            "CarWinRate": car_win_rate,
            "CarPodiumRate": car_pod_rate,
            "CarPointsRate": car_pts_rate,
            "CircuitAvgCompoundRank": cir_avg_comp,
            "CircuitAvgStints": total_stints,
            "CircuitAvgTrackTemp": cir_avg_temp,
            "ThreatScoreAhead": threat_score,
            "MaxThreatAhead": max_threat,
            "FreshTyre": 1,
            "FinalPosition": current_pos,
            "IsWin": 0,
            "StintCompound": "SOFT",
            "NextPitLap": float(total_laps),
        }

        row_spark = spark.createDataFrame([row_dict])
        for cat in CAT_FEATURES:
            row_spark = row_spark.withColumn(cat, F.lit(driver if cat=="Driver" else (team if cat=="TeamName" else circuit)))

        _pipe_cols = ["CompoundLabel","features_raw","features"] + [c+"_idx" for c in CAT_FEATURES] + [c+"_ohe" for c in CAT_FEATURES]
        for c in _pipe_cols:
            if c in row_spark.columns: row_spark = row_spark.drop(c)
            
        row_proc = preproc_model.transform(row_spark)

        # Predict compound
        if SPARK_XGB: comp_pred = int(m1_model.transform(row_proc).select("prediction").first()[0])
        else:
            arr = np.array([row_proc.select("features").first()[0].toArray()])
            comp_pred = int(m1.predict(arr)[0])

        # Predict compound probabilities instead of just the top greedy choice
        if SPARK_XGB:
            prob_vec = m1_model.transform(row_proc).select("probability").first()[0].toArray()
        else:
            arr = np.array([row_proc.select("features").first()[0].toArray()])
            prob_vec = m1.predict_proba(arr)[0]
            
        # Create a sorted list of preferred compounds
        comp_prefs = [(COMPOUND_IDX_TO_NAME.get(i, "SOFT"), p) for i, p in enumerate(prob_vec)]
        comp_prefs.sort(key=lambda x: x[1], reverse=True)
        
        # --- PHYSICS HEURISTICS: Compound Selection Context ---
        # 1-Stop races (total_stints == 2) require highly durable tyres. 
        # If we are doing a 1-stop, and this is the first stint, we almost certainly need a HARD or MEDIUM.
        # If this is a 2+ stop, SOFTs are more viable.
        
        valid_compounds = [c[0] for c in comp_prefs if c[0] in ["SOFT", "MEDIUM", "HARD"]]
        
        # Filter 1: Pirelli rule - must use 2 different compounds
        if stint_num == total_stints and total_stints > 1:
            used_compounds = set([s["compound"] for s in strategy])
            # If we've only used one compound type so far, we MUST pick something different
            if len(used_compounds) == 1:
                used_c = list(used_compounds)[0]
                valid_compounds = [c for c in valid_compounds if c != used_c]
                
        # Filter 2: Back-to-back compounds are inefficient (usually you change compound when you pit)
        if len(valid_compounds) > 1 and prev_compound in valid_compounds:
            valid_compounds.remove(prev_compound)
            
        # Filter 3: 1-Stop Strategy Durability
        # If doing a 1-stop race (2 stints total), using SOFT is extremely risky unless it's a very short stint.
        if total_stints <= 2:
            # If the ML suggested SOFT, but it's a 1 stop, force it towards HARD/MEDIUM
            if valid_compounds[0] == "SOFT":
                # Push HARD or MEDIUM up
                hard_med = [c for c in valid_compounds if c in ["HARD", "MEDIUM"]]
                if hard_med:
                    valid_compounds = hard_med + [c for c in valid_compounds if c not in hard_med]
            
            # If this is a 1-stop, one of the stints MUST be HARD to survive.
            # If stint 1 wasn't HARD, stint 2 must be HARD (or vice versa).
            if stint_num == 2 and "HARD" not in [s["compound"] for s in strategy]:
                if "HARD" in valid_compounds:
                    valid_compounds.remove("HARD")
                    valid_compounds.insert(0, "HARD")

        # Pick the best valid compound
        compound = valid_compounds[0] if valid_compounds else "SOFT"

        # Predict next pit lap
        if stint_num == total_stints:
            next_pit = total_laps  # Last stint goes to the end
        else:
            if SPARK_XGB: p_lap = int(round(float(m2_model.transform(row_proc).select("prediction").first()[0])))
            else:     
                arr = np.array([row_proc.select("features").first()[0].toArray()])
                p_lap = int(round(float(m2.predict(arr)[0])))
            
            # --- PHYSICS CONSTRAINT ---
            max_laps_for_compound = TYRE_LIFE_LIMITS.get(compound, 30)
            
            p_lap = min(p_lap, current_lap + max_laps_for_compound) # Respect tyre limit
            p_lap = max(p_lap, current_lap + 5)                     # Minimum practical stint
            next_pit = p_lap
            
            # If the remaining laps exceed remaining limits, extend this stint as much as possible safely
            remaining_laps = total_laps - next_pit
            remaining_stints = total_stints - stint_num
            if remaining_laps > remaining_stints * max(TYRE_LIFE_LIMITS.values()):
                # Unreachable; extend this stint up to its absolute physical maximum
                next_pit = min(current_lap + max_laps_for_compound, total_laps - remaining_stints*5)

        strategy.append({
            "stint": stint_num,
            "lap_start": current_lap,
            "lap_end": next_pit - 1 if next_pit < total_laps else total_laps,
            "compound": compound,
            "laps_on_tyre": (next_pit - 1 if next_pit < total_laps else total_laps) - current_lap + 1
        })

        prev_compound = compound
        current_lap = next_pit

    # Predict win probability based on full context
    wp_dict = dict(row_dict)
    wp_dict["StintCompound"] = strategy[0]["compound"]
    wp_row = spark.createDataFrame([wp_dict])
    for cat in CAT_FEATURES: wp_row = wp_row.withColumn(cat, F.lit(driver if cat=="Driver" else (team if cat=="TeamName" else circuit)))
    for c in _pipe_cols:
        if c in wp_row.columns: wp_row = wp_row.drop(c)
    wp_proc = preproc_model.transform(wp_row)

    if SPARK_XGB: raw_win_prob = float(m3_model.transform(wp_proc).select("probability").first()[0][1])
    else:         raw_win_prob = float(m3.predict_proba(np.array([wp_proc.select("features").first()[0].toArray()]))[0, 1])

    # --- CALIBRATE WIN PROBABILITY ---
    # ML raw probabilities for rare events (winning is 1/20) are naturally low globally.
    # An XGBoost raw probability of 0.08 is actually very high. 
    # We calibrate it contextually based on their grid position and the threat ahead.
    
    # Baseline expected baseline for the grid position vs historical converting rates
    # P1 wins ~40%, P2 ~25%, P3 ~15%
    grid_baseline = 0.40 if quali_position == 1 else (0.25 if quali_position == 2 else (0.15 if quali_position == 3 else max(0.01, 0.10 - quali_position*0.01)))
    
    # If the driver has a high historical win rate themselves, boost it
    driver_factor = drv_win_rate * 2.0  # Max is ~0.30, *2 = 0.60
    
    # If the cars ahead are weak (ThrestScoreAhead is low compared to grid position), boost it.
    # If cars ahead are strong (e.g. ThreatScore ahead > 0.40), penalize it.
    threat_penalty = max(0, threat_score - drv_win_rate) * 0.5
    
    # Combine M3's raw signal with grid heuristics to form a realistic normalized probability
    calibrated_prob = (raw_win_prob * 3.0) + grid_baseline + (driver_factor * 0.5) - threat_penalty
    calibrated_prob = max(0.001, min(0.999, calibrated_prob))
    
    win_prob = calibrated_prob

    if verbose:
        print(f"\n{'='*65}")
        print(f"  STRATEGY PREDICTION ({num_pit_stops}-Stop)")
        print(f"  Driver: {driver}  |  Team: {team}")
        print(f"  Circuit: {circuit}  |  Year: {year}  |  Grid: P{quali_position}")
        print(f"  Threat Ahead: {threat_score:.2f} (Max: {max_threat:.2f})")
        print(f"{'='*65}")
        print(f"  {'Stint':<8} {'Laps':<18} {'Compound':<14} {'Tyre Life'}")
        print(f"  {'-'*55}")
        for s in strategy:
            lap_range = f"Lap {s['lap_start']} – {s['lap_end']}"
            print(f"  {s['stint']:<8} {lap_range:<18} {s['compound']:<14} {s['laps_on_tyre']} laps")
        print(f"{'='*65}")
        print(f"  Win Probability  : {win_prob*100:.1f}%")
        print(f"{'='*65}")

    return {"strategy": strategy, "win_probability": win_prob}


## 13. Example Strategy Predictions

In [0]:
# Example 1 – Hamilton at Monza 2023 from P1
result = predict_strategy(
    driver="HAM",
    team="Mercedes",
    year=2023,
    circuit="Monza",
    quali_position=1,
    total_laps=53,
    num_pit_stops=1,
)


In [0]:
# Example 2 – Verstappen at Silverstone 2022 from P2
result2 = predict_strategy(
    driver="VER",
    team="Red Bull Racing",
    year=2022,
    circuit="Silverstone",
    quali_position=2,
    total_laps=52,
    num_pit_stops=2,
)


In [0]:
# Example 3 – Leclerc at Monaco 2022 from P1
result3 = predict_strategy(
    driver="LEC",
    team="Ferrari",
    year=2022,
    circuit="Monaco",
    quali_position=1,
    total_laps=78,
    num_pit_stops=1,
)


In [0]:
# ── Visualise strategy as a tyre stint chart ──────────────────────────
COMPOUND_COLORS = {
    "SOFT": "#e74c3c",
    "MEDIUM": "#f39c12",
    "HARD": "#95a5a6",
    "INTERMEDIATE": "#2ecc71",
    "WET": "#2980b9"
}

def plot_strategy(result_dict, driver, circuit, year, total_laps, ax=None):
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(14, 2.5))
    for s in result_dict["strategy"]:
        color = COMPOUND_COLORS.get(s["compound"], "#9b59b6")
        ax.barh(0, s["laps_on_tyre"], left=s["lap_start"]-1, height=0.6,
                color=color, edgecolor="white", linewidth=1.5)
        ax.text(
            s["lap_start"] - 1 + s["laps_on_tyre"]/2, 0,
            f"{s['compound'][0]} {s['laps_on_tyre']}",
            ha="center", va="center", fontsize=9, fontweight="bold", color="white"
        )
    ax.set_xlim(0, total_laps)
    ax.set_ylim(-0.5, 0.5)
    ax.set_xlabel("Lap Number")
    ax.set_yticks([])
    win_pct = result_dict["win_probability"] * 100
    ax.set_title(f"{driver} | {circuit} {year}  —  Win Prob: {win_pct:.1f}%", fontsize=11, fontweight="bold")
    if standalone:
        plt.tight_layout(); plt.show()

fig, axes = plt.subplots(3, 1, figsize=(14, 8))
plot_strategy(result,  "HAM",  "Monza",      2023, 53, ax=axes[0])
plot_strategy(result2, "VER",  "Silverstone", 2022, 52, ax=axes[1])
plot_strategy(result3, "LEC",  "Monaco",      2022, 78, ax=axes[2])
plt.tight_layout(); plt.show()


## 14. Full Evaluation Summary

In [0]:
summary_data = {
    "Model": ["M1 Compound (Val)", "M1 Compound (Test)",
              "M2 Pit Lap (Val)",  "M2 Pit Lap (Test)",
              "M3 Win Prob (Val)", "M3 Win Prob (Test)"],
}

# Collect metrics
from sklearn.metrics import roc_auc_score
metrics_rows = []
for name, y_t, y_p, kind in [
    ("M1 Compound — Validation", y_true_m1_v.astype(int), y_pred_m1_v.astype(int), "clf"),
    ("M1 Compound — Test",       y_true_m1_t.astype(int), y_pred_m1_t.astype(int), "clf"),
    ("M2 Pit Lap — Validation",  y_true_m2_v, y_pred_m2_v, "reg"),
    ("M2 Pit Lap — Test",        y_true_m2_t, y_pred_m2_t, "reg"),
    ("M3 Win Prob — Validation",  y_true_m3_v, y_prob_m3_v,  "bin"),
    ("M3 Win Prob — Test",        y_true_m3_t, y_prob_m3_t,  "bin"),
]:
    if kind == "clf":
        r = {"Model": name,
             "Accuracy": f"{accuracy_score(y_t,y_p):.3f}",
             "F1 Macro": f"{f1_score(y_t,y_p,average='macro',zero_division=0):.3f}",
             "MAE (laps)": "—", "RMSE": "—", "R²": "—",
             "AUC-ROC": "—", "Avg Precision": "—"}
    elif kind == "reg":
        r = {"Model": name, "Accuracy": "—", "F1 Macro": "—",
             "MAE (laps)": f"{mean_absolute_error(y_t,y_p):.2f}",
             "RMSE": f"{np.sqrt(mean_squared_error(y_t,y_p)):.2f}",
             "R²": f"{r2_score(y_t,y_p):.3f}",
             "AUC-ROC": "—", "Avg Precision": "—"}
    else:
        from sklearn.metrics import average_precision_score
        r = {"Model": name, "Accuracy": "—", "F1 Macro": "—",
             "MAE (laps)": "—", "RMSE": "—", "R²": "—",
             "AUC-ROC": f"{roc_auc_score(y_t,y_p):.3f}",
             "Avg Precision": f"{average_precision_score(y_t,y_p):.3f}"}
    metrics_rows.append(r)

summary_df = pd.DataFrame(metrics_rows).set_index("Model")
display(summary_df)


## 15. Save Models

In [0]:
import os, joblib

# -------------------------
# PERSIST FOR DOWNSTREAM USE (phase_4.ipynb + the Streamlit strategy app)
# -------------------------
# Everything below used to save only to "../models/", a path relative to
# this notebook's own workspace location. That's fine for immediate reuse
# within this same notebook session, but nothing else can reach it: a
# separate notebook (phase_4.ipynb) has a different relative root, and a
# Databricks App runs in its own isolated container with no access to
# workspace-relative paths at all. Unity Catalog Volumes are the one
# storage layer both a notebook and an App can reach via the same absolute
# path (see phase_1.ipynb's cache fix for the same lesson) -- so the model
# artifacts now go there instead. driver_profile/constructor_profile/
# circuit_profile are small, structured, and queried by key, so those go
# to Unity Catalog tables rather than the Volume.
STRATEGY_VOLUME_DIR = "/Volumes/workspace/default/f1_data/strategy_models"
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.f1_data")
os.makedirs(STRATEGY_VOLUME_DIR, exist_ok=True)

# Preprocessing pipeline (Spark ML PipelineModel -- plain directory write,
# not SQLite, so the Volume mount handles it fine).
preproc_model.write().overwrite().save(f"{STRATEGY_VOLUME_DIR}/preproc_pipeline")
print("Preprocessing pipeline saved to Volume (OK)")

# XGBoost models (SPARK_XGB is hardcoded False in the environment-setup
# cell, so this notebook always takes the plain-xgboost branch).
if SPARK_XGB:
    m1_model.save(f"{STRATEGY_VOLUME_DIR}/m1_compound_classifier")
    m2_model.save(f"{STRATEGY_VOLUME_DIR}/m2_pitlap_regressor")
    m3_model.save(f"{STRATEGY_VOLUME_DIR}/m3_win_classifier")
else:
    m1.save_model(f"{STRATEGY_VOLUME_DIR}/m1_compound_classifier.json")
    m2.save_model(f"{STRATEGY_VOLUME_DIR}/m2_pitlap_regressor.json")
    m3.save_model(f"{STRATEGY_VOLUME_DIR}/m3_win_classifier.json")

# Label map
import json as _json
with open(f"{STRATEGY_VOLUME_DIR}/compound_label_map.json", "w") as f:
    _json.dump({str(k): v for k, v in COMPOUND_IDX_TO_NAME.items()}, f)

print("All strategy models saved to", STRATEGY_VOLUME_DIR)

# Profile tables -- Unity Catalog tables, not the Volume, since these are
# structured lookups keyed by (Driver, Year) / (TeamName, Year) / Circuit,
# not opaque model files.
driver_profile.write.mode("overwrite").saveAsTable("workspace.default.f1_strategy_driver_profile")
constructor_profile.write.mode("overwrite").saveAsTable("workspace.default.f1_strategy_constructor_profile")
circuit_profile.write.mode("overwrite").saveAsTable("workspace.default.f1_strategy_circuit_profile")
print("Profile tables saved: f1_strategy_driver_profile, f1_strategy_constructor_profile, f1_strategy_circuit_profile")

# Local relative-path saves kept too, purely as a same-session convenience
# (e.g. eyeballing/downloading files from this notebook's own run) -- not
# what phase_4.ipynb or the app actually load from; those use the Volume
# and UC tables above.
os.makedirs("../models", exist_ok=True)
preproc_model.write().overwrite().save("../models/preproc_pipeline")
if not SPARK_XGB:
    m1.save_model("../models/m1_compound_classifier.json")
    m2.save_model("../models/m2_pitlap_regressor.json")
    m3.save_model("../models/m3_win_classifier.json")
with open("../models/compound_label_map.json", "w") as f:
    _json.dump({str(k): v for k, v in COMPOUND_IDX_TO_NAME.items()}, f)

print("(Local session-only copies also written to ../models/)")

---
## Summary

| Model | Purpose | Primary Metric |
|-------|---------|---------------|
| **M1** XGBoost Classifier | Predict tyre compound per stint | Accuracy / F1-macro |
| **M2** XGBoost Regressor | Predict lap number to pit | MAE / RMSE |
| **M3** XGBoost Classifier | Predict win probability | AUC-ROC |

### Training Parameters Rationale

| Parameter | Driver Profile | Constructor Profile |
|-----------|---------------|---------------------|
| `DriverAvgPos` | Raw finishing skill | — |
| `DriverWinRate` | Race-winning ability | — |
| `DriverConsistency` | Crash/error rate | — |
| `DriverAvgQuali` | One-lap pace | — |
| `CarAvgPos` | — | Car competitiveness |
| `CarWinRate` | — | Car race-winning ability |
| `CarPodiumRate` | — | Car front-running pace |
| `CarAvgQuali` | — | Raw car speed |

> **Combined performance** = `DriverProfile × CarProfile` — both surfaces naturally through the XGBoost features.
